# NB00 — Baseline WCC (BTCS)
**Budgeted Temporal Case Segmentation for AML Transaction Monitoring**
Andre da Costa Silva | ITA | 2026

## Objetivo
Reproduzir os números do artigo ICEIS 2026 com erro < 1%.
Este notebook é a **fundação** de toda a pesquisa — não avance sem validar os resultados.

**Fluxo:** Setup → Definir funções → Carregar dados → Rodar WCC → Validar vs ICEIS

In [ ]:
# ============================================================
# CÉLULA 1 — Setup completo: imports, paths, verificação
# ============================================================
import os, sys, subprocess, time, math, contextlib
from pathlib import Path

# ── Instalar dependências ──────────────────────────────────
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

_pip("numpy", "pandas", "scikit-learn", "matplotlib", "networkx", "psutil", "tqdm")

try:
    import torch
except ImportError:
    _pip("torch", "torchvision", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/cu121")
    import torch

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
from IPython.display import display

# ── Device ────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE}")

# ── Google Drive (Colab) ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"[WARN] Drive: {e}")

BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path(".").resolve()

# ── Caminhos confirmados em março/2026 ────────────────────
# AML100k: modelo 'SAGE', seed 42
AML100K_BASE  = BASE / "DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k"
AML100K_ARTIF = AML100K_BASE / "artifacts"
AML100K_PROBS = AML100K_BASE / "results/probs_v4"
AML100K_MODEL = "SAGE"
AML100K_SEED  = 42

# AML1M: modelo 'GraphSAGE', seed 44
AML1M_BASE    = BASE / "DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M"
AML1M_ARTIF   = AML1M_BASE / "artifacts"
AML1M_PROBS   = AML1M_BASE / "results_aml1m_graphsage_only/probs_v4"
AML1M_MODEL   = "GraphSAGE"
AML1M_SEED    = 44

# Saída dos resultados
BTCS_OUTPUT_DIR = BASE / "GrafosGNN/results/nb00_baseline"
BTCS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Verificar arquivos ────────────────────────────────────
print("\n=== Verificação de arquivos ===")
checks = [
    ("AML100k cache", AML100K_ARTIF / "edge_data_v4_clean.pt"),
    ("AML100k probs", AML100K_PROBS / f"{AML100K_MODEL}_seed{AML100K_SEED}_test.npz"),
    ("AML1M   cache", AML1M_ARTIF   / "edge_data_v4_clean.pt"),
    ("AML1M   probs", AML1M_PROBS   / f"{AML1M_MODEL}_seed{AML1M_SEED}_test.npz"),
]
ok_all = True
for label, path in checks:
    ok = path.exists()
    ok_all = ok_all and ok
    print(f"  {'✓' if ok else '✗'}  {label}: {path}")

if ok_all:
    print("\n  ✅ Todos os arquivos encontrados!")
else:
    print("\n  ❌ Algum arquivo não encontrado — verifique o Drive.")


## Funções: métricas, WCC, carregar dados

In [ ]:
# ============================================================
# CÉLULA 2 — Definição de TODAS as funções
# (executar uma vez antes de rodar qualquer experimento)
# ============================================================

# ── measure_resources ─────────────────────────────────────
@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    mem0 = proc.memory_info().rss / 1024**2
    t0   = time.perf_counter()
    res  = {}
    yield res
    res['time_s'] = time.perf_counter() - t0
    res['ram_mb']  = proc.memory_info().rss / 1024**2 - mem0

# ── Union-Find ────────────────────────────────────────────
def _uf_init(n):
    return np.arange(n, dtype=np.int64), np.zeros(n, dtype=np.int64)

def _find(parent, a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a

def _union(parent, rank, a, b):
    ra, rb = _find(parent, a), _find(parent, b)
    if ra == rb: return
    if rank[ra] < rank[rb]: parent[ra] = rb
    elif rank[ra] > rank[rb]: parent[rb] = ra
    else: parent[rb] = ra; rank[ra] += 1

# ── _build_wcc ────────────────────────────────────────────
def _build_wcc(src_sel, dst_sel, sel_idx, full_eval_edges, min_nodes=3):
    nodes_all = np.unique(np.concatenate([src_sel, dst_sel]))
    node_map  = {int(n): i for i, n in enumerate(nodes_all)}
    N = len(nodes_all)
    us = np.fromiter((node_map[int(x)] for x in src_sel), dtype=np.int64)
    vs = np.fromiter((node_map[int(x)] for x in dst_sel), dtype=np.int64)
    parent, rank = _uf_init(N)
    for a, b in zip(us, vs):
        _union(parent, rank, int(a), int(b))
    comp     = np.array([_find(parent, i) for i in range(N)], dtype=np.int64)
    _, comp_ids = np.unique(comp, return_inverse=True)
    sizes    = np.bincount(comp_ids)
    valid    = np.where(sizes >= min_nodes)[0]
    gid_remap = {int(c): i for i, c in enumerate(valid)}
    max_node  = int(nodes_all.max()) + 1
    gid_of    = -np.ones(max_node, dtype=np.int64)
    for node_id, local_idx in node_map.items():
        c = int(comp_ids[local_idx])
        if c in gid_remap:
            gid_of[node_id] = gid_remap[c]
    n_cases = len(valid)
    if n_cases == 0:
        return []
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_cases)]
    for i, (u, v) in enumerate(zip(src_sel, dst_sel)):
        g = gid_of[int(u)] if int(u) < max_node else -1
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel_idx[i]))
            cases[g]['nodes'].update([int(u), int(v)])
    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    for i, (u, v) in enumerate(zip(src_full, dst_full)):
        gu = gid_of[int(u)] if int(u) < max_node else -1
        gv = gid_of[int(v)] if int(v) < max_node else -1
        if gu >= 0 and gu == gv:
            cases[gu]['induced_edges'].append(i)
    return cases

# ── build_cases ───────────────────────────────────────────
def build_cases(ranked_edges, full_eval_edges, method='wcc', k=0.01, params=None):
    if params is None: params = {}
    scores = np.asarray(ranked_edges['scores'], dtype=float)
    src    = np.asarray(ranked_edges['src'],    dtype=np.int64)
    dst    = np.asarray(ranked_edges['dst'],    dtype=np.int64)
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]
    if method == 'wcc':
        return _build_wcc(src[sel], dst[sel], sel, full_eval_edges,
                          min_nodes=params.get('min_nodes', 3))
    raise NotImplementedError(f'Método "{method}" não implementado ainda.')

# ── evaluate_cases ────────────────────────────────────────
def evaluate_cases(cases, ground_truth, ranked_edges, k):
    y_full  = np.asarray(ground_truth['y_full'], dtype=int)
    scores  = np.asarray(ranked_edges['scores'], dtype=float)
    y_test  = np.asarray(ranked_edges['y'],      dtype=int)
    E_test  = len(y_test)
    pos_te  = int(y_test.sum())
    K       = max(1, int(math.ceil(k * E_test)))
    if not cases:
        return {m: np.nan for m in ['n_cases','coverage','purity_induced','cr_at_k',
                'edges_per_case_median','edges_per_case_p95','edges_per_case_max',
                'e_ind_over_ek','ocr_b100','ocr_b500','ocr_b1000']}
    ind_sizes  = np.array([len(c['induced_edges']) for c in cases])
    pos_ind    = sum(int(y_full[c['induced_edges']].sum()) for c in cases)
    coverage   = float(pos_ind / max(pos_te, 1))
    purity     = float(pos_ind / max(ind_sizes.sum(), 1))
    pos_sel    = sum(int(y_test[c['seed_edges']].sum()) for c in cases)
    recall     = float(pos_sel / max(pos_te, 1))
    cr_at_k    = float(recall / max(coverage, 1e-12)) if coverage > 0 else np.nan
    return {
        'n_cases':               len(cases),
        'coverage':              coverage,
        'purity_induced':        purity,
        'cr_at_k':               cr_at_k,
        'recall_in_cases':       recall,
        'edges_per_case_median': float(np.median(ind_sizes)),
        'edges_per_case_p95':    float(np.percentile(ind_sizes, 95)),
        'edges_per_case_max':    float(ind_sizes.max()),
        'e_ind_total':           int(ind_sizes.sum()),
        'e_ind_over_ek':         float(ind_sizes.sum() / max(K, 1)),
        'ocr_b100':              float((ind_sizes > 100).mean()),
        'ocr_b500':              float((ind_sizes > 500).mean()),
        'ocr_b1000':             float((ind_sizes > 1000).mean()),
    }

# ── load_dataset_artifacts ────────────────────────────────
def load_dataset_artifacts(artif_dir, probs_dir, model='GraphSAGE', seed=44, dataset_name=''):
    artif_dir = Path(artif_dir)
    probs_dir = Path(probs_dir)

    npz_path = probs_dir / f'{model}_seed{seed}_test.npz'
    if not npz_path.exists():
        raise FileNotFoundError(f'Probs não encontrado: {npz_path}')
    npz   = np.load(npz_path)
    p_te  = np.asarray(npz['p'], dtype=float)
    y_te  = np.asarray(npz['y'], dtype=int)
    print(f'[{dataset_name}] Scores: {len(p_te):,} arestas de teste, '
          f'{y_te.sum():,} positivas ({100*y_te.mean():.2f}%)')

    cache_path = artif_dir / 'edge_data_v4_clean.pt'
    if not cache_path.exists():
        raise FileNotFoundError(f'Cache não encontrado: {cache_path}')
    cache  = torch.load(cache_path, map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']

    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()

    src_te = ei_all[0, te_idx]
    dst_te = ei_all[1, te_idx]
    assert len(p_te) == len(src_te), f'Mismatch scores/arestas: {len(p_te)} vs {len(src_te)}'

    print(f'[{dataset_name}] Cache: {ei_all.shape[1]:,} arestas totais, {len(te_idx):,} no teste')
    return (
        {'scores': p_te, 'y': y_te, 'src': src_te, 'dst': dst_te},
        {'src': src_te, 'dst': dst_te},
        {'y_full': y_te},
    )

print('✓ Todas as funções definidas.')


## Carregar artefatos do Stage 1

In [ ]:
# ============================================================
# CÉLULA 3 — Carregar artefatos (rodar após Célula 1 e 2)
# ============================================================
print("Carregando AML100k...")
ranked_100k, full_100k, gt_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS,
    model=AML100K_MODEL, seed=AML100K_SEED,
    dataset_name='AML100k'
)

print("\nCarregando AML1M (pode demorar ~30s)...")
ranked_1m, full_1m, gt_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS,
    model=AML1M_MODEL, seed=AML1M_SEED,
    dataset_name='AML1M'
)

print("\n✅ Dados carregados! Pode rodar a Célula 4.")


## Rodar WCC Baseline para k ∈ {1%, 2%, 5%, 10%}

In [ ]:
# ============================================================
# CÉLULA 4 — WCC Baseline (B0)
# ============================================================
KS         = [0.01, 0.02, 0.05, 0.10]
WCC_PARAMS = {'min_nodes': 3}
DATASETS   = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]

rows = []
for ds_name, ranked, full, gt in DATASETS:
    print(f'\n=== {ds_name} ===')
    for k in KS:
        with measure_resources() as res:
            cases   = build_cases(ranked, full, method='wcc', k=k, params=WCC_PARAMS)
            metrics = evaluate_cases(cases, gt, ranked, k)
        row = {'dataset': ds_name, 'method': 'B0_WCC', 'k%': f'{int(k*100)}%',
               'k_frac': k, **metrics, 'time_s': res['time_s'], 'ram_mb': res['ram_mb']}
        rows.append(row)
        print(f"  k={int(k*100)}%: #casos={metrics['n_cases']:,} | "
              f"coverage={metrics['coverage']:.3f} | purity={metrics['purity_induced']:.4f} | "
              f"CR@k={metrics['cr_at_k']:.3f} | E_ind/E_k={metrics['e_ind_over_ek']:.2f}x | "
              f"{res['time_s']:.1f}s")

df_results = pd.DataFrame(rows)
print('\n✅ WCC Baseline concluído!')


## Resultados e Validação vs ICEIS 2026

In [ ]:
# ============================================================
# CÉLULA 5 — Tabela principal + verificação vs ICEIS
# ============================================================
display_cols = ['dataset','k%','n_cases','coverage','purity_induced',
                'cr_at_k','edges_per_case_median','edges_per_case_p95',
                'e_ind_over_ek','time_s']
df_disp = df_results[display_cols].copy()
df_disp.columns = ['Dataset','k%','#Cases','Coverage','Purity',
                   'CR@k','Edges/Case (med)','Edges/Case (p95)','|E_ind|/|E_k|','Time (s)']
print('=== WCC Baseline (B0) ===')
display(df_disp.round(4))

# ── Referência ICEIS — preencha com seus números do paper ──
iceis_reference = {
    ('AML1M', '1%'):  {'n_cases': 10546, 'e_ind_over_ek': 1.64},
    ('AML1M', '5%'):  {'n_cases':  4549, 'e_ind_over_ek': 3.82},
    ('AML1M', '10%'): {'n_cases':  1388, 'e_ind_over_ek': 3.60},
}

print('\n=== Verificação vs ICEIS (erro deve ser < 1%) ===')
all_ok = True
for (ds, kpct), ref in iceis_reference.items():
    row = df_results[(df_results['dataset']==ds) & (df_results['k%']==kpct)]
    if row.empty:
        print(f'  ⚠  {ds} k={kpct}: não encontrado'); continue
    r = row.iloc[0]
    for metric, expected in ref.items():
        got = float(r[metric])
        err = abs(got - expected) / max(abs(expected), 1e-9)
        ok  = err < 0.01
        if not ok: all_ok = False
        print(f"  {'✓' if ok else '✗'} {ds} k={kpct} {metric}: got={got:.3f}, "
              f"expected={expected:.3f}, err={err*100:.2f}%")

if all_ok:
    print('\n✅ BASELINE VALIDADO — pode avançar para nb01!')
else:
    print('\n❌ Algum número diverge > 1% — verifique paths e seeds.')


## OCR-B — Fração de casos grandes (problema motivador do BTCS)

In [ ]:
# ============================================================
# CÉLULA 6 — OCR-B
# ============================================================
ocr_cols = ['dataset','k%','n_cases','ocr_b100','ocr_b500','ocr_b1000','edges_per_case_max']
df_ocr   = df_results[ocr_cols].copy()
df_ocr.columns = ['Dataset','k%','#Cases','OCR(B=100)','OCR(B=500)','OCR(B=1000)','Max edges/case']
print('=== OCR-B: Fração de casos acima do limiar B ===')
display(df_ocr.round(4))
print('\n→ OCR-B alto (especialmente k=5% e 10%) é o problema que o BTCS resolve.')
print('  Meta BTCS: reduzir OCR-B em > 60% (critério mínimo) ou > 85% (critério forte).')


## Export CSV + LaTeX

In [ ]:
# ============================================================
# CÉLULA 7 — Salvar resultados
# ============================================================
out_csv = BTCS_OUTPUT_DIR / 'b0_wcc_baseline_results.csv'
df_results.to_csv(out_csv, index=False)
print(f'✓ CSV salvo: {out_csv}')

out_tex = BTCS_OUTPUT_DIR / 'b0_wcc_baseline_table.tex'
df_disp.round(4).to_latex(out_tex, index=False, escape=False)
print(f'✓ LaTeX salvo: {out_tex}')

print('\n=== Roadmap ===')
print('1. ✅ nb00 — WCC reproduzido (este notebook)')
print('2. ⏳ nb01 — B1 (Temporal-WCC), B2, B3')
print('3. ⏳ nb02 — Grafo Lk + Leiden + Contextualização (BTCS)')
print('4. ⏳ nb03 — Ablações sistemáticas')
print('5. ⏳ nb04 — AMLworld + Libra Bank')
